# Segmentation performance against a hand-corrected ground truth

Compares the segmentations of one `.zarr` (written by `macroseg.baseline`) with masks corrected by hand
in the Cellpose GUI (`02_cellpose_gui.ipynb`). Reads the store `cfg.zarr` and the annotations in
`$MACROSEG_ANNOTATIONS/round_<NN>/<name>/`; on the Mac these are the copies made by notebook 02 (paths in `.env`).

Cells are matched one to one by their overlap (IoU, intersection over union). At each IoU threshold:

| Measure | Definition | Says |
|---|---|---|
| AP | matched / (matched + false cells + missed cells) | overall quality (Cellpose's definition) |
| precision | matched / segmented cells | how many segmented cells are real |
| recall | matched / true cells | how many true cells are found |
| recall on hand-drawn cells | recall on the cells drawn in the GUI only | the cells the starting masks got wrong |

IoU 0.5 mostly says whether a cell is found; 0.75 and 0.9 whether its outline is right.

**Bias:** the ground truth was made by correcting one segmentation, so that one scores high
(its untouched cells are identical). A fair comparison needs an image whose masks were not used for training.
CPU only; a few seconds per segmentation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from skimage.measure import regionprops_table
from spatialdata import read_zarr

from macroseg import paths
from macroseg.accuracy import load_cellpose_masks, match_cells, plot_ap_curve, score_segmentations
from macroseg.annotation import image_folder, load_annotations, summarize_rounds
from macroseg.config import load_config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_test_crop.yaml")
ANNOTATIONS = paths.path("MACROSEG_ANNOTATIONS")
ROUND = 1

## 0. Annotation rounds
Every image of every round (`annotation.json`), then one line per round. `start_score`: the starting masks
scored against their correction; it should rise from round to round as the model improves.


In [ ]:
annotations = load_annotations(ANNOTATIONS)
annotations

In [ ]:
summarize_rounds(annotations)

## 1. Scores
One column per segmentation of the store, against the corrected masks of `ROUND`.


In [ ]:
seg_file = image_folder(ANNOTATIONS, ROUND, cfg.name) / f"{cfg.name}_seg.npy"
sdata = read_zarr(cfg.zarr)
truth = load_cellpose_masks(seg_file)
hand_drawn = np.flatnonzero(np.load(seg_file, allow_pickle=True).item()["ismanual"]) + 1  # GUI cell IDs
segmentations = {step.labels: sdata.labels[step.labels].data.compute() for step in cfg.segmentation}

table = score_segmentations(truth, segmentations, thresholds=[0.5, 0.75, 0.9], truth_ids=hand_drawn)
table.rename(columns=lambda c: c.replace("subset_", "hand_drawn_")).T.round(3)

## 2. AP and recall against the IoU threshold
A curve that stays high to the right means precise outlines.


In [ ]:
curve = score_segmentations(truth, segmentations, thresholds=np.round(np.arange(0.5, 1.0, 0.05), 2))
plot_ap_curve(curve, "AP");
plot_ap_curve(curve, "recall");

## 3. Cells to look at
True cells with the worst match in one segmentation (`iou` 0 = missed). `y`, `x`: centre in pixels,
to find them in napari or the Cellpose GUI.


In [ ]:
name = "membrane_dapi"
centres = pd.DataFrame(regionprops_table(truth, properties=("label", "centroid")))
centres = centres.rename(columns={"label": "truth_ID", "centroid-0": "y", "centroid-1": "x"}).round().astype(int)
matches = match_cells(truth, segmentations[name]).merge(centres)
matches["hand_drawn"] = matches["truth_ID"].isin(hand_drawn)
matches.sort_values("iou").head(15)